In [ ]:
from collections.abc import Sequence
from pathlib import Path

import numpy as np
import pandas as pd




In [ ]:
# --- Step 1: Define paths and constants ---

cwd = Path.cwd().resolve()


def find_release_root(start: Path) -> Path:
    base = start if start.is_dir() else start.parent
    sentinel_dirs = ("Step0_Data", "Step9_FutureProjection")
    for candidate in [base, *base.parents]:
        if all((candidate / name).is_dir() for name in sentinel_dirs):
            return candidate
        nested_project = candidate / "Global-solid-waste-prediction"
        if all((nested_project / name).is_dir() for name in sentinel_dirs):
            return nested_project
    raise FileNotFoundError("Could not locate Global-solid-waste-prediction repository root from the current working directory")


release_root = find_release_root(cwd)
output_dir = release_root / "Step9_FutureProjection" / "2_Results"
historical_path = release_root / "Step8_HistoricalCompletion" / "2_Results" / "0_historical_panel_completed.csv"
super_panel_path = output_dir / "1_super_panel_1990_2100.csv"
corrected_panel_path = output_dir / "2_super_panel_1990_2100.csv"
factor_table_path = output_dir / "2_anchor_bias_factors.csv"
audit_table_path = output_dir / "2_bias_correction_audit.csv"
ANCHOR_YEAR = 2024
FUTURE_START_YEAR = 2025
BRIDGE_END_YEAR = 2030
FUTURE_SOURCE = "bias_corrected_trend_bridge_2024_2030"
WASTE_SPECS = {
    "AW": {"t": "AW_t", "pred": "AW_pred", "final": "AW_final", "source": "AW_source"},
    "CDW": {"t": "CDW_t", "pred": "CDW_pred", "final": "CDW_final", "source": "CDW_source"},
    "IW": {"t": "IW_t", "pred": "IW_pred", "final": "IW_final", "source": "IW_source"},
    "MSW": {"t": "MSW_t", "pred": "MSW_pred", "final": "MSW_final", "source": "MSW_source"},
}
KEY_COLS = ["Country Code", "Year", "Scenario"]
LONG_SHARED_COLS = [
    "Country Code",
    "Year",
    "Scenario",
    "IncomeGroup",
    "Region",
    "SP.POP.TOTL",
    "NY.GDP.MKTP.PP.KD",
    "NY.GDP.PCAP.PP.KD",
    "EN.POP.DNST",
]




In [ ]:
# --- Step 2: Build conversion helpers ---


def validate_required_columns(frame: pd.DataFrame, required_cols: list[str], label: str) -> None:
    missing_cols = [col for col in required_cols if col not in frame.columns]
    if missing_cols:
        raise RuntimeError(f"{label} is missing required columns: {missing_cols}")


def build_country_name_lookup(hist_df: pd.DataFrame) -> pd.DataFrame:
    validate_required_columns(hist_df, ["Country Code", "Country Name"], "historical_df")
    country_frame = hist_df.loc[:, ["Country Code", "Country Name"]].dropna().copy()
    country_frame = country_frame.groupby("Country Code", as_index=False)["Country Name"].first()
    return country_frame.reset_index(drop=True)


def wide_panel_to_long(panel_df: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for waste_flag, cols in WASTE_SPECS.items():
        required_cols = LONG_SHARED_COLS + [cols["t"], cols["pred"], cols["final"], cols["source"]]
        validate_required_columns(panel_df, required_cols, f"panel_df[{waste_flag}]")
        part = panel_df[required_cols].copy()
        part["WasteFlag"] = waste_flag
        part.columns = LONG_SHARED_COLS + ["Observed_Raw", "Predicted_Raw", "Final_Raw", "Source", "WasteFlag"]
        parts.append(part)
    return pd.concat(parts, ignore_index=True)


def corrected_long_to_wide(base_future: pd.DataFrame, corrected_long: pd.DataFrame) -> pd.DataFrame:
    validate_required_columns(base_future, KEY_COLS, "base_future")
    future_panel = base_future.copy()
    for waste_flag, cols in WASTE_SPECS.items():
        waste_part = corrected_long.loc[
            corrected_long["WasteFlag"] == waste_flag,
            KEY_COLS + ["Corrected_Raw"],
        ].copy()
        if waste_part.empty:
            raise RuntimeError(f"Corrected future rows are empty for waste flag: {waste_flag}")
        new_pred_col = f"{cols['pred']}__new"
        waste_part.columns = KEY_COLS + [new_pred_col]
        future_panel = future_panel.merge(waste_part, on=KEY_COLS, how="left")
        if bool(future_panel[new_pred_col].isna().any()):
            sample = future_panel.loc[future_panel[new_pred_col].isna(), KEY_COLS].head(10).to_dict("records")
            raise RuntimeError(f"Missing corrected values for {waste_flag}: {sample}")
        future_panel[cols["pred"]] = pd.to_numeric(future_panel.pop(new_pred_col), errors="coerce")
        future_panel[cols["final"]] = future_panel[cols["pred"]]
        future_panel[cols["source"]] = FUTURE_SOURCE
    return future_panel




In [ ]:
# --- Step 3: Build factor table and trend bridge helpers ---


def build_anchor_factor_table(panel_long: pd.DataFrame, country_lookup: pd.DataFrame) -> pd.DataFrame:
    def rounded_nunique(series: pd.Series) -> int:
        numeric = pd.Series(pd.to_numeric(series, errors="coerce"), index=series.index, dtype="float64")
        return int(numeric.round(10).dropna().nunique())

    anchor_rows = panel_long.loc[panel_long["Year"] == ANCHOR_YEAR].copy()
    scenario_check = (
        anchor_rows.groupby(["Country Code", "WasteFlag"], as_index=False)
        .agg(
            SourceNunique=("Source", lambda s: s.dropna().astype(str).nunique()),
            PredNunique=("Predicted_Raw", rounded_nunique),
            FinalNunique=("Final_Raw", rounded_nunique),
        )
    )
    bad = scenario_check.loc[
        (scenario_check["SourceNunique"] > 1)
        | (scenario_check["PredNunique"] > 1)
        | (scenario_check["FinalNunique"] > 1)
    ]
    if not bad.empty:
        sample = bad.head(10).to_dict("records")
        raise RuntimeError(f"Historical rows are not scenario-invariant at anchor year: {sample}")

    dedup = anchor_rows.sort_values("Scenario").drop_duplicates(subset=["Country Code", "WasteFlag"]).copy()
    dedup["Source_2024"] = dedup["Source"].astype(str)
    dedup["Pred_2024"] = pd.to_numeric(dedup["Predicted_Raw"], errors="coerce")
    dedup["Final_2024"] = pd.to_numeric(dedup["Final_Raw"], errors="coerce")
    dedup["HasObservedAnchor"] = dedup["Source_2024"].eq("observed")
    dedup["HasValidPredAnchor"] = dedup["Pred_2024"].gt(0)
    dedup["R_2024"] = np.where(
        dedup["HasObservedAnchor"] & dedup["HasValidPredAnchor"],
        dedup["Final_2024"] / dedup["Pred_2024"],
        np.nan,
    )
    dedup["BiasClass"] = np.select(
        [
            dedup["R_2024"].isna(),
            (dedup["R_2024"] - 1.0).abs() >= 0.05,
        ],
        ["not_available", "explicit_bias"],
        default="mild_bias",
    )
    dedup["FactorStatus"] = np.select(
        [
            dedup["HasObservedAnchor"] & dedup["HasValidPredAnchor"],
            dedup["HasObservedAnchor"] & (~dedup["HasValidPredAnchor"]),
        ],
        ["applied", "invalid"],
        default="not_available",
    )
    dedup["Reason"] = np.select(
        [
            dedup["HasObservedAnchor"] & dedup["HasValidPredAnchor"],
            dedup["HasObservedAnchor"] & (~dedup["HasValidPredAnchor"]),
        ],
        ["applied", "invalid_pred_2024"],
        default="no_observed_anchor_2024",
    )
    dedup = dedup.merge(country_lookup, on="Country Code", how="left")
    return dedup[
        [
            "Country Code",
            "Country Name",
            "WasteFlag",
            "Source_2024",
            "Pred_2024",
            "Final_2024",
            "R_2024",
            "BiasClass",
            "FactorStatus",
            "Reason",
        ]
    ]


def coerce_year_value(value: object) -> int:
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating, str)):
        return int(value)
    raise RuntimeError(f"Unexpected year value: {value}")


def bridge_future_series_from_2024(anchor_value: float, stage1_future: pd.Series) -> tuple[pd.Series, int, str]:
    future_series = stage1_future.sort_index().astype(float).copy()
    if future_series.empty:
        return future_series, ANCHOR_YEAR, "empty_future"

    bridge_years = [int(year) for year in future_series.index.tolist() if int(year) <= BRIDGE_END_YEAR]
    if not bridge_years:
        return future_series, int(future_series.index.max()), "pass_through"

    target_year = bridge_years[-1]
    target_value = float(future_series.loc[target_year])
    direction = float(np.sign(target_value - float(anchor_value)))
    previous_value = float(anchor_value)

    for idx, year in enumerate(bridge_years, start=1):
        weight = idx / len(bridge_years)
        candidate = float(anchor_value) + (target_value - float(anchor_value)) * weight
        if direction > 0:
            candidate = max(candidate, previous_value)
            candidate = min(candidate, target_value)
        elif direction < 0:
            candidate = min(candidate, previous_value)
            candidate = max(candidate, target_value)
        else:
            candidate = float(anchor_value)
        future_series.loc[year] = max(candidate, 0.0)
        previous_value = float(future_series.loc[year])

    return future_series.clip(lower=0.0), target_year, "trend_bridge_2024_2030"




In [ ]:
# --- Step 4: Apply factor stage and trend bridge ---


def apply_bias_and_bridge(future_long: pd.DataFrame, factor_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    merged = future_long.merge(
        factor_df[
            [
                "Country Code",
                "Country Name",
                "WasteFlag",
                "Source_2024",
                "Final_2024",
                "R_2024",
                "FactorStatus",
                "Reason",
            ]
        ],
        on=["Country Code", "WasteFlag"],
        how="left",
    )
    merged["FactorStatus"] = merged["FactorStatus"].fillna("not_available")
    merged["Reason"] = merged["Reason"].fillna("missing_factor_row")
    merged["Predicted_Raw"] = pd.to_numeric(merged["Predicted_Raw"], errors="coerce")
    merged["Stage1_Raw"] = np.where(
        merged["FactorStatus"].eq("applied"),
        merged["Predicted_Raw"] * pd.to_numeric(merged["R_2024"], errors="coerce"),
        merged["Predicted_Raw"],
    )

    corrected_parts = []
    audit_rows = []

    grouped = merged.groupby(["Scenario", "Country Code", "WasteFlag"], sort=True)
    for group_key, block in grouped:
        if not isinstance(group_key, Sequence) or len(group_key) != 3:
            raise RuntimeError(f"Unexpected group key: {group_key}")
        scenario_key, country_code_key, waste_flag_key = tuple(group_key)
        scenario = str(scenario_key)
        country_code = str(country_code_key)
        waste_flag = str(waste_flag_key)
        part = block.sort_values("Year").copy()
        part["Year"] = part["Year"].map(coerce_year_value)

        anchor_value = float(pd.to_numeric(part["Final_2024"].iloc[0], errors="coerce"))
        stage1_series = pd.Series(
            pd.to_numeric(part["Stage1_Raw"], errors="coerce").to_numpy(dtype=float),
            index=part["Year"].to_numpy(dtype=int),
            dtype=float,
        )
        corrected_series, target_year, bridge_status = bridge_future_series_from_2024(
            anchor_value=anchor_value,
            stage1_future=stage1_series,
        )
        corrected_map = corrected_series.to_dict()
        part["Corrected_Raw"] = part["Year"].map(corrected_map).astype(float)
        corrected_parts.append(part)

        raw_2025 = float(part.loc[part["Year"] == 2025, "Predicted_Raw"].iloc[0]) if bool((part["Year"] == 2025).any()) else np.nan
        stage1_2025 = float(part.loc[part["Year"] == 2025, "Stage1_Raw"].iloc[0]) if bool((part["Year"] == 2025).any()) else np.nan
        corrected_2025 = float(part.loc[part["Year"] == 2025, "Corrected_Raw"].iloc[0]) if bool((part["Year"] == 2025).any()) else np.nan
        target_stage1 = float(corrected_series.loc[target_year]) if target_year in corrected_series.index else np.nan
        bridge_direction = np.sign(target_stage1 - anchor_value) if np.isfinite(target_stage1) else np.nan

        audit_rows.append(
            {
                "Scenario": scenario,
                "Country Code": country_code,
                "Country Name": str(part["Country Name"].iloc[0]) if "Country Name" in part.columns else np.nan,
                "WasteFlag": waste_flag,
                "Source_2024": str(part["Source_2024"].iloc[0]),
                "FactorStatus": str(part["FactorStatus"].iloc[0]),
                "Reason": str(part["Reason"].iloc[0]),
                "Anchor_2024": anchor_value,
                "Raw_2025": raw_2025,
                "Stage1_2025": stage1_2025,
                "Corrected_2025": corrected_2025,
                "TargetYear": int(target_year),
                "TargetValue": target_stage1,
                "BridgeStatus": bridge_status,
                "BridgeDirection": float(bridge_direction) if np.isfinite(bridge_direction) else np.nan,
            }
        )

    corrected_df = pd.concat(corrected_parts, ignore_index=True)
    audit_df = pd.DataFrame(audit_rows).sort_values(["Scenario", "Country Code", "WasteFlag"]).reset_index(drop=True)
    return corrected_df, audit_df




In [ ]:
# --- Step 5: Export corrected outputs ---

historical = pd.read_csv(historical_path, usecols=["Country Code", "Country Name"])
super_panel = pd.read_csv(super_panel_path)
country_lookup = build_country_name_lookup(historical)
panel_long = wide_panel_to_long(super_panel)

factor_df = build_anchor_factor_table(panel_long, country_lookup)
future_long = panel_long.loc[panel_long["Year"] >= FUTURE_START_YEAR].copy()
corrected_long, audit_df = apply_bias_and_bridge(future_long, factor_df)

history_panel = super_panel.loc[super_panel["Year"] < FUTURE_START_YEAR].copy()
future_panel = super_panel.loc[super_panel["Year"] >= FUTURE_START_YEAR].copy()
future_corrected_wide = corrected_long_to_wide(future_panel, corrected_long)
corrected_panel = pd.concat([history_panel, future_corrected_wide], ignore_index=True)
corrected_panel = corrected_panel.sort_values(["Scenario", "Country Code", "Year"]).reset_index(drop=True)

raw_hist = super_panel.loc[super_panel["Year"] < FUTURE_START_YEAR].sort_values(["Scenario", "Country Code", "Year"]).reset_index(drop=True)
corrected_hist = corrected_panel.loc[corrected_panel["Year"] < FUTURE_START_YEAR].sort_values(["Scenario", "Country Code", "Year"]).reset_index(drop=True)
if not raw_hist.equals(corrected_hist):
    raise RuntimeError("Historical rows changed during trend-bridge correction")
if len(corrected_panel) != len(super_panel):
    raise RuntimeError("Corrected panel row count does not match the input super panel")

factor_df = factor_df.sort_values(["WasteFlag", "Country Code"]).reset_index(drop=True)
factor_df.to_csv(factor_table_path, index=False)
audit_df.to_csv(audit_table_path, index=False)
corrected_panel.to_csv(corrected_panel_path, index=False)

summary_cols = ["AW_final", "CDW_final", "IW_final", "MSW_final"]
for scenario in ["SSP1", "SSP2", "SSP3", "SSP4", "SSP5"]:
    total_2024 = float(corrected_panel.loc[(corrected_panel["Scenario"] == scenario) & (corrected_panel["Year"] == 2024), summary_cols].sum().sum() / 1e9)
    total_2025 = float(corrected_panel.loc[(corrected_panel["Scenario"] == scenario) & (corrected_panel["Year"] == 2025), summary_cols].sum().sum() / 1e9)
    print(f"{scenario}: 2024={total_2024:.6f} -> 2025={total_2025:.6f}")

print(corrected_panel.shape)
print(factor_df.groupby(["WasteFlag", "FactorStatus"], dropna=False).size().to_string())
print(audit_df.head().to_string(index=False))
